# NOPP_d2: v1 vs v2 product comparison

Side-by-side of two upcast velocity products:

- **A = v1 baseline** (`L2_grid1m.nc`, upcasts subset): AHRS attitude, WWcorr_beam motion correction, **no sail term** — its Jun–Oct horizontal velocities carry an O(0.05–0.1 m/s) lean-azimuth bias.
- **B = v2s2** (`L2up_v2s2_grid1m.nc`): buoyant-ascent motion model — dp/dt vertical + sail (along-wire travel on the leaning wire, validated by wire conservation: span = 499.9·cos(tilt), r = 0.95), LP-accel tilt, tilt-compensated compass heading, per-bin Doppler SEM.

Both carry per-cast QC: `ahrs_error_deg` (fault statistic), and B adds `attitude_source` / `heading_source`.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

P = "/Users/drew/NOPP/pyNOPP/nopp1-california/S100430A038_NOPP_d2/processed"
full = xr.open_dataset(f"{P}/nopp_d2_sig1000_L2_grid1m.nc")
A = full.where(full.cast_direction == 1, drop=True)          # v1, upcasts only
B = xr.open_dataset(f"{P}/nopp_d2_sig1000_L2up_v2s2_grid1m.nc")

common, ia, ib = np.intersect1d(A.time.values, B.time.values, return_indices=True)
A, B = A.isel(cast=ia), B.isel(cast=ib)
t = pd.to_datetime(common)
z = A.depth.values
err = B.ahrs_error_deg.values
print(f"matched upcasts: {common.size}   ({t[0].date()} .. {t[-1].date()})")
print("A:", full.attrs.get("motion_correction"))
print("B:", B.attrs.get("motion_correction"))
print(f"B attitude: {int((B.attitude_source.values==1).sum())} lp-tilt, "
      f"{int((B.heading_source.values==1).sum())} mag-heading, "
      f"faulted casts (err>=15): {int((err>=15).sum())}")

## 1. Velocity sections — A, B, and their difference
The difference panel is dominated by the sail term after June (the vehicle leaned 8–13° once the currents strengthened).

In [ ]:
def sections(var, vmax=0.25, dmax=0.12):
    fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, constrained_layout=True)
    for ax, (fld, ttl, vm, cmap) in zip(axes, [
            (A[var].values, f"{var} — A (v1 baseline)", vmax, "RdBu_r"),
            (B[var].values, f"{var} — B (v2s2)", vmax, "RdBu_r"),
            (B[var].values - A[var].values, "B − A", dmax, "PuOr_r")]):
        pc = ax.pcolormesh(t, z, fld, vmin=-vm, vmax=vm, cmap=cmap, shading="nearest")
        ax.set_ylim(510, 0); ax.set_ylabel("depth (m)"); ax.set_title(ttl, loc="left")
        fig.colorbar(pc, ax=ax, pad=0.01, label="m/s")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    return fig

sections("velE");

In [ ]:
sections("velN");

## 2. Monthly mean profiles
Dashed = A (v1), solid = B (v2s2). The deep (150–500 m) Jun–Oct reversal — apparent weak poleward flow in A becoming equatorward, surface-aligned flow in B — is the sail bias being removed.

In [ ]:
months = pd.period_range(t[0], t[-1], freq="M")
fig, axes = plt.subplots(2, 4, figsize=(14, 8), sharey=True, constrained_layout=True)
for ax, mo in zip(axes.ravel(), months):
    sel = (t.to_period("M") == mo)
    if sel.sum() < 20:
        ax.set_visible(False); continue
    for var, col in (("velE", "tab:blue"), ("velN", "tab:red")):
        ax.plot(np.nanmean(A[var].values[:, sel], axis=1), z, "--", color=col, lw=1.2)
        ax.plot(np.nanmean(B[var].values[:, sel], axis=1), z, "-", color=col, lw=1.8,
                label=var)
    ax.axvline(0, color="0.6", lw=0.5)
    ax.set_title(str(mo)); ax.set_xlim(-0.25, 0.25)
    ax.set_xlabel("m/s")
axes[0, 0].set_ylim(510, 0); axes[0, 0].set_ylabel("depth (m)")
axes[0, 0].legend(loc="lower right", fontsize=8)
fig.suptitle("monthly mean profiles — dashed A (v1) vs solid B (v2s2); velE blue, velN red");

## 3. Depth-mean current: speed and direction per cast

In [ ]:
def depth_mean(DS, z0, z1):
    m = (z >= z0) & (z < z1)
    return (np.nanmean(DS.velE.values[m], axis=0), np.nanmean(DS.velN.values[m], axis=0))

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, constrained_layout=True)
for row, (z0, z1) in enumerate(((0, 150), (150, 500))):
    for DS, name, col in ((A, "A (v1)", "0.55"), (B, "B (v2s2)", "tab:blue")):
        uE, uN = depth_mean(DS, z0, z1)
        spd = pd.Series(np.hypot(uE, uN), index=t).rolling("2D").median()
        azi = (np.rad2deg(np.arctan2(uE, uN)) % 360)
        axes[row, 0].plot(t, spd, color=col, lw=1.2, label=name)
        axes[row, 1].scatter(t, azi, s=2, color=col, alpha=0.5, label=name)
    axes[row, 0].set_ylabel(f"{z0}-{z1} m\nspeed (m/s)")
    axes[row, 1].set_ylabel("direction (deg true)"); axes[row, 1].set_ylim(0, 360)
axes[0, 0].legend(); axes[0, 1].legend(markerscale=4)
axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.suptitle("depth-mean current per cast (speed: 2-day rolling median)");

## 4. Single-cast profiles with the SEM envelope
Pick any cast index `j` (sorted by time). The shaded band is B's per-bin Doppler-only standard error (`velE_sem` etc.) — a floor, not a total error bar. Default: the worst AHRS-fault cast, where A and B should differ most.

In [ ]:
def plot_cast(j):
    fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True, constrained_layout=True)
    for ax, var in zip(axes, ("velE", "velN", "velU")):
        ax.plot(A[var].values[:, j], z, color="0.55", lw=1, label="A (v1)")
        vb, sb = B[var].values[:, j], B[f"{var}_sem"].values[:, j]
        ax.plot(vb, z, color="tab:blue", lw=1.4, label="B (v2s2)")
        ax.fill_betweenx(z, vb - sb, vb + sb, color="tab:blue", alpha=0.25, lw=0)
        ax.axvline(0, color="0.7", lw=0.5); ax.set_title(var); ax.set_xlabel("m/s")
    axes[0].set_ylim(510, 0); axes[0].set_ylabel("depth (m)")
    axes[0].legend(fontsize=8)
    fig.suptitle(f"cast {j}  {t[j]:%Y-%m-%d %H:%M}   ahrs_error {err[j]:.1f} deg, "
                 f"heading={'mag' if B.heading_source.values[j]==1 else 'ahrs'}")

j_worst = int(np.nanargmax(err))
plot_cast(j_worst)

In [ ]:
# velU sanity: the un-fitted arbiter. B should be tighter around zero,
# most visibly on the faulted casts.
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for DS, name, col in ((A, "A (v1)", "0.55"), (B, "B (v2s2)", "tab:blue")):
    mu = np.nanmean(DS.velU.values, axis=0)
    ax.hist(mu, bins=np.linspace(-0.05, 0.05, 101), histtype="step",
            color=col, lw=1.5, label=f"{name}  (|median| {abs(np.nanmedian(mu)):.4f})")
ax.set_xlabel("cast-mean velU (m/s)"); ax.set_ylabel("casts"); ax.legend()
ax.set_title("cast-mean vertical velocity");

## 5. Two-week window browser
Slide `start` to choose which two weeks to inspect; `vmax` sets the color scale. Left column A (v1), right column B (v2s2); rows velE / velN. (Requires a live kernel for the sliders.)

In [ ]:
import ipywidgets as W

starts = pd.date_range(t[0].normalize(), t[-1] - pd.Timedelta(days=14), freq="D")
date_slider = W.SelectionSlider(options=[(d.strftime("%Y-%m-%d"), d) for d in starts],
                                description="start", continuous_update=False,
                                layout=W.Layout(width="75%"))
vmax_slider = W.FloatSlider(value=0.25, min=0.05, max=0.5, step=0.05,
                            description="vmax", continuous_update=False)

def two_weeks(start, vmax):
    sel = (t >= start) & (t < start + pd.Timedelta(days=14))
    if sel.sum() < 5:
        print("no casts in this window (duty-cycle gap)"); return
    tw = t[sel]
    fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    for row, var in enumerate(("velE", "velN")):
        for col, (DS, name) in enumerate(((A, "A (v1)"), (B, "B (v2s2)"))):
            ax = axes[row, col]
            pc = ax.pcolormesh(tw, z, DS[var].values[:, sel], vmin=-vmax, vmax=vmax,
                               cmap="RdBu_r", shading="nearest")
            ax.set_title(f"{var} — {name}", loc="left", fontsize=10)
    axes[0, 0].set_ylim(510, 0)
    for ax in axes[:, 0]:
        ax.set_ylabel("depth (m)")
    axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.colorbar(pc, ax=axes, pad=0.01, label="m/s")
    n_fault = int((err[sel] >= 15).sum())
    fig.suptitle(f"{start:%Y-%m-%d} + 14 d   ({sel.sum()} casts, {n_fault} AHRS-faulted)")
    plt.show()

W.interact(two_weeks, start=date_slider, vmax=vmax_slider);